# In-distribution finite-size validation

Compare normalized risk against aspect ratio and trial MSE against budget. The study uses six budgets, 11 requested aspect ratios on $[0.1,10]$, $\lambda=10^{-4}$, ten trials and seed 17.

Run all cells to load the included simulation summaries and theoretical predictions, redraw the figure, and save its PDF. Set `RECOMPUTE=True` to regenerate synthetic training trials and theory. Plotting and export never start missing calculations.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import sys
import importlib
import inspect
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# This notebook accepts only this bundle or its notebooks/ directory.
# It never discovers a parent research checkout or imports its cache.
candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
BUNDLE_ROOT = next((p for p in candidates
    if (p / 'dms').is_dir() and (p / 'notebooks' / '01_landscapes.ipynb').is_file()), None)
if BUNDLE_ROOT is None:
    raise RuntimeError('Start Jupyter in the reproducibility bundle or its notebooks/ directory.')
if 'dms' in sys.modules:
    loaded = Path(sys.modules['dms'].__file__).resolve().parent
    if loaded != BUNDLE_ROOT / 'dms':
        raise RuntimeError('Restart the kernel: dms was imported from another checkout.')
sys.path.insert(0, str(BUNDLE_ROOT))
PROJECT_ROOT = BUNDLE_ROOT  # bundle-local numerical source root
OUTPUT_DIR = BUNDLE_ROOT / 'output'
plt.rcParams.update({'axes.labelsize': 11, 'axes.labelweight': 'bold',
                     'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11})
importlib.invalidate_caches()
for module_name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory', 'dms.optimization', 'dms.jobs', 'dms.ridge_diagnostics', 'dms.exploration', 'dms.result_cache', 'dms'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

from dms import DMSTTheory, project_stieltjes_evaluator
from dms.exploration import SweepParameters, parameter_sweep_cases
from dms.jobs import finite_size_job, gr_job_sizes
from dms.simulation import moment_feasibility
from dms.result_cache import ResultCache

theory = DMSTTheory(project_stieltjes_evaluator())
CASES, FIGURES = {}, {}
CACHE_OPTIONS = dict(directory=BUNDLE_ROOT / '.cache' / 'finite_size',
                     enabled=True, verbose=True)
def cache():
    return ResultCache(**CACHE_OPTIONS)
def sources(*names):
    return [BUNDLE_ROOT / 'dms' / (name + '.py') for name in names]
THEORY_SOURCES = sources('config','stieltjes','theory','ridge_diagnostics','landscape_slices')
SIM_SOURCES = sources('config','simulation','jobs')

from tools.portable_results import load_results, assert_settings_match

# False: reproduce from the included arrays. True: calculate from the inputs below.
RECOMPUTE = False


## 1. Exact inputs

`L_ref=100` converts normalized means into raw moments once; the raw moments remain fixed across all sizes. Budget B selects `n=max(2,floor(B/sqrt(r)))` and `L=b*floor(B*sqrt(r)/(q*b))`, with b the denominator of eta (b=1 for this in-distribution study). Reported points use actual `r=qL/n`.

Unlike the main incremental simulation, this convergence job uses one sequential NumPy `default_rng(seed+i)` stream per budget at its zero-based list position i. It generates one landscape for each size/r and then its 10 training trials. Keep budget and r ordering fixed for the same realizations. Its cache stores whole results rather than extendable raw-trial prefixes.


In [ ]:
CASES["fig1_matched"] = dict(parameters=SweepParameters(
    q=20,sigma=1.,G2=100.,f=.1,ftilde=.1,eta=0.,
    F=.4,FTest=.4,Fneq=0.,rhoneq=0.,m=0.1,mneq=0.,mneqtilde=0.),
    reference_L=100,mean_convention="normalized",
    plot=dict(title=""),compute={})

def case_config(key):
    spec=CASES[key];base=spec["parameters"]
    mt=spec.get("test_mismatch_mean")
    case,=parameter_sweep_cases(base,"mneqtilde",[base.mneqtilde if mt is None else mt],
        mean_convention=spec["mean_convention"],reference_L=spec["reference_L"])
    return case.config

# Existing in-distribution controls and their numerical cache identity are retained.
CONVERGENCE = dict(run=True, case="fig1_matched", budgets=[100, 250, 500, 1000, 2000, 4000],
                   r_values=np.geomspace(.1,10.,11), ridge_lambda=1e-4, trials=10, seed=17)
CONVERGENCE_THEORY_POINTS = 201
CONVERGENCE_FORCE_SIMULATION = False
CONVERGENCE_FORCE_THEORY = False

CONVERGENCE_DATA = None  # populated by the theory stage or cache-only redraw


## 2. Independent cached numerical stages
These helpers never start simulation while restoring results for theory/plotting. Missing inputs identify the stage to run.


In [ ]:
def convergence_spec():
    spec = deepcopy(CONVERGENCE)
    if spec['case'] not in CASES:
        raise KeyError(f"Unknown finite-size case: {spec['case']}")
    cfg = case_config(spec['case'])
    inputs = dict(config=cfg.as_dict(), **{k:v for k,v in spec.items()
                                          if k not in ('run','case')})
    return spec, cfg, inputs


def convergence_preflight():
    spec, cfg, _ = convergence_spec()
    rows = []
    for budget in spec['budgets']:
        for requested_r in spec['r_values']:
            point = dict(budget=int(budget), requested_r=float(requested_r))
            try:
                ((nn, ll),) = gr_job_sizes([requested_r], cfg, budget)
            except ValueError as error:
                rows.append(dict(point, n=None, L=None, realized_r=None,
                                 feasible=False, messages=(str(error),)))
                continue  # report every invalid size, including budgets with zero rounded L
            report = moment_feasibility(int(ll), cfg)
            rows.append(dict(point, n=int(nn), L=int(ll),
                             realized_r=float(cfg.q*ll/nn), feasible=bool(report.feasible),
                             messages=report.messages))
    return rows


def published_convergence():
    data, metadata = load_results(BUNDLE_ROOT/'data'/'finite_size', restore_records=True)
    settings = dict(simulation=convergence_spec()[2], theory_points=CONVERGENCE_THEORY_POINTS)
    assert_settings_match(metadata['settings'], settings)
    return data


def convergence_simulation(*, compute=False, force=False):
    if not RECOMPUTE:
        data = published_convergence()
        return data['result'], data['ids']['simulation']
    spec, cfg, inputs = convergence_spec()
    if compute:
        errors = [row for row in convergence_preflight() if not row['feasible']]
        if errors:
            raise ValueError(errors)
    # Separate simulation cache; theory uses the realized aspect ratios.
    return cache().get('convergence', inputs,
        lambda: finite_size_job(spec['r_values'], spec['ridge_lambda'], cfg,
            n_runs=spec['trials'], budgets=spec['budgets'], seed=spec['seed']),
        sources=SIM_SOURCES, compute=compute, force=force)


def convergence_theory(*, compute=False, force=False):
    global CONVERGENCE_DATA
    if not RECOMPUTE:
        CONVERGENCE_DATA = published_convergence()
        return CONVERGENCE_DATA
    spec, _, settings = convergence_spec()
    result, sim_id = convergence_simulation(compute=False)
    realized = result.data[:, :, 0]
    theory_points = CONVERGENCE_THEORY_POINTS
    dense_r = np.geomspace(realized.min(), realized.max(), theory_points)
    inputs = dict(config=result.config.as_dict(), ridge_lambda=result.ridge_lambda, r_values=dense_r)
    dense, dense_id = cache().get('convergence_theory', inputs,
        lambda: np.array([theory.risk(result.ridge_lambda, float(r), result.config) for r in dense_r]),
        sources=THEORY_SOURCES, compute=compute, force=force)
    sampled, sampled_id = cache().get('convergence_sample_theory',
        dict(config=result.config.as_dict(), ridge_lambda=result.ridge_lambda, r_values=realized),
        lambda: np.array([[theory.risk(result.ridge_lambda, float(r), result.config)
                           for r in row] for row in realized]),
        sources=THEORY_SOURCES, compute=compute, force=force)
    CONVERGENCE_DATA = dict(result=result, dense_r=dense_r, dense=dense, sampled=sampled,
                study='in_distribution', case=spec['case'], trials=int(spec['trials']),
                settings=deepcopy(settings), theory_points=int(theory_points),
                ids=dict(simulation=sim_id, dense=dense_id, sampled=sampled_id))
    return CONVERGENCE_DATA


## 3. Synthetic training trials

Features are independent-site categorical sequences around the training reference; Gaussian training noise has variance sigma². Ridge fitting empirically centers features/labels and leaves the intercept unpenalized. Conditional test risk is integrated exactly over the independent-site test distribution, with no sampled test set and with additive test-noise sigma² omitted. Trials resample training sequences and noise while keeping one constructed landscape fixed at each size/r. Error bars are one SE of the trial mean using `ddof=1`.


In [ ]:
# In-distribution simulation stage; the existing whole-result caches remain reusable.
if CONVERGENCE['run']:
    CONVERGENCE_RESULT, CONVERGENCE_SIM_ID = convergence_simulation(
        compute=True, force=CONVERGENCE_FORCE_SIMULATION)
    print('In-distribution case:', CONVERGENCE['case'])
    display([dict(budget=int(budget), requested_r=float(requested), n=int(nn), L=int(ll),
                  realized_r=float(point[0]))
             for budget, job in zip(CONVERGENCE_RESULT.budgets, CONVERGENCE_RESULT.jobs)
             for requested, (nn,ll), point in zip(job.requested_r, job.sizes, job.data[0])])
else:
    print("In-distribution simulation disabled; plotting can still use matching caches.")


## 4. Theory at realized aspect ratios
Theory is evaluated at every realized r, so integer-size rounding does not enter the measured discrepancy. A separate dense curve is used only for plotting.


In [ ]:
# In-distribution theory stage only; its simulation must already exist on disk.
CONVERGENCE_DATA = convergence_theory(
    compute=True, force=CONVERGENCE_FORCE_THEORY)


## 5. Figure construction and MSE definition

The right panel reports $\frac1T\sum_{t=1}^{T}(R_t-R_{\rm th})^2/R_\infty^2$, **not prediction risk itself** and not merely the squared error of the trial mean. From the saved mean and standard error it equals $[(\overline R-R_{\rm th})^2+(T-1)\mathrm{SE}^2]/R_\infty^2$. We draw no MSE error bars because its sampling uncertainty cannot be recovered from these two aggregate moments. The selected requested aspect ratios `.1,1,10` are matched exactly within floating-point tolerance; no interpolation is used.

All figure construction is editable in the following cell and consumes arrays only.


In [ ]:
def draw_convergence_figure(data, style):
    result = data['result']
    scale = result.r_infinity
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError('Finite-size validation needs a positive finite R_infinity.')
    selected_rs = np.asarray(style['mse_r_values'], dtype=float)
    if selected_rs.ndim != 1 or not selected_rs.size or len(np.unique(selected_rs)) != len(selected_rs):
        raise ValueError('mse_r_values must be a nonempty list of distinct requested r values.')
    # Select by requested r, not nearest realized r (which changes with budget).
    selected_columns = []
    for selected_r in selected_rs:
        columns = []
        for job in result.jobs:
            matches = np.flatnonzero(np.isclose(job.requested_r, selected_r, rtol=1e-10, atol=0.))
            if len(matches) != 1:
                raise ValueError(f'r={selected_r:g} is not a unique stored requested r. '
                                 'Select an existing value or update CONVERGENCE and run simulation/theory.')
            columns.append(int(matches[0]))
        selected_columns.append(columns)
    # SE = sample_std(ddof=1)/sqrt(T). Thus (T-1)*SE**2 is the empirical
    # trial variance with divisor T; adding squared bias gives the trial MSE.
    mse = ((result.data[:,:,1] - data['sampled'])**2
           + (data['trials'] - 1) * result.data[:,:,2]**2) / scale**2
    fig, axes = plt.subplots(1, 2, figsize=(style['width'], style['height']),
                             dpi=style['dpi'], layout='constrained')
    fig.get_layout_engine().set(**style['layout_padding'])
    colors = plt.get_cmap(style['cmap'])(np.linspace(*style['color_span'], len(result.budgets)))
    for budget, points, color in zip(result.budgets, result.data, colors):
        kwargs = dict(style['simulation']) | style['budget_styles'].get(int(budget), {})
        kwargs.setdefault('color', color)
        label = style['budget_labels'].get(int(budget), fr'$B={int(budget)}$')
        axes[0].errorbar(points[:,0], points[:,1]/scale, yerr=points[:,2]/scale,
                         label=label, **kwargs)
    axes[0].plot(data['dense_r'], data['dense']/scale, **style['theory'])
    r_colors = plt.get_cmap(style['r_cmap'])(np.linspace(*style['r_color_span'], len(selected_rs)))
    order = np.argsort(result.budgets)
    for selected_r, columns, color in zip(selected_rs, selected_columns, r_colors):
        kwargs = dict(style['mse_line']) | style['r_styles'].get(float(selected_r), {})
        kwargs.setdefault('color', color)
        label = style['r_labels'].get(float(selected_r), fr'$r={selected_r:g}$')
        values = mse[np.arange(len(result.budgets)), columns]
        if style['yscales'][1] == 'log' and np.any(values <= 0):
            plt.close(fig)
            raise ValueError('MSE contains zero: use a linear or symlog right y axis; no positive floor is imposed.')
        axes[1].plot(result.budgets[order], values[order], label=label, **kwargs)
    labels = [r'Risk $R/R_\infty$',
              r'MSE $\langle(R_{\rm sim}-R_{\rm th})^2\rangle/R_\infty^2$']
    for i, ax in enumerate(axes):
        ax.set(xscale=style['xscales'][i], yscale=style['yscales'][i],
               xlabel=style['xlabels'][i], ylabel=style['ylabels'][i] or labels[i])
        if style['xlims'][i] is not None: ax.set_xlim(style['xlims'][i])
        if style['xticks'][i] is not None:
            ax.set_xticks(style['xticks'][i])
            if i == 1: ax.set_xticklabels([f'{v:g}' for v in style['xticks'][i]])
        if style['ylims'][i] is not None: ax.set_ylim(style['ylims'][i])
        if style['yticks'][i] is not None: ax.set_yticks(style['yticks'][i])
        ax.tick_params(labelsize=style['tick_size'])
        for label in (ax.xaxis.label, ax.yaxis.label):
            label.set_fontsize(style['label_size']); label.set_fontweight('bold')
        ax.grid(style['grid'])
        if style['panel_labels']:
            ax.text(*style['panel_label_xy'], chr(97+i), transform=ax.transAxes,
                    fontsize=style['panel_label_size'], fontweight='bold')
        if style['show_legends'][i]:
            ax.legend(**style['legends'][i])
    if style['annotation'] is not False:
        exponent = np.log10(result.ridge_lambda)
        ridge_text = (fr'10^{{{int(round(exponent))}}}' if np.isclose(exponent, round(exponent))
                      else f'{result.ridge_lambda:g}')
        text = style['annotation'] or fr'$\lambda={ridge_text}$'
        axes[0].text(*style['annotation_xy'], text, transform=axes[0].transAxes,
                     **style['annotation_style'])
    return fig


def plot_convergence(style, *, force=False):
    data = convergence_theory(compute=False)  # cache-only, including sampled-r theory
    helpers = (draw_convergence_figure, plot_convergence)
    try:
        implementation = [inspect.getsource(f) for f in helpers]
    except (OSError, TypeError):
        implementation, force = [], True
    fig, _ = cache().get('convergence_figure',
        dict(study=data['study'], results=data['ids'], style=style, rcparams={k:repr(v) for k,v in plt.rcParams.items()}),
        lambda: draw_convergence_figure(data, style), implementation=implementation, force=force)
    FIGURES['convergence'] = fig
    display(fig); plt.close(fig)
    return fig


## 6. Plot-only controls and redraw
The lambda annotation is separate from the legend. Both panels retain independent axis/legend controls; all sizes, fonts, colours, lines, and markers are editable here.


In [ ]:
# Plot-only controls. The two panels have independent axes and curve legends.
CONVERGENCE_PLOT = dict(
    width=7.5, height=3.4, dpi=140,
    cmap='cividis', color_span=(.15,.85),       # left: budgets
    r_cmap='viridis', r_color_span=(.15,.85),  # right: requested aspect ratios
    mse_r_values=[.1,1.,10.],                 # must be present in stored simulation r_values
    label_size=11, tick_size=11, panel_label_size=12,
    panel_labels=True, panel_label_xy=(-.10,1.03),
    layout_padding=dict(w_pad=.03,h_pad=.03,wspace=.08,hspace=.03),
    xlabels=[r'Aspect ratio $r=qL/n$', r'Budget $B$'], ylabels=[None,None],
    xscales=['log','log'], yscales=['linear','log'],
    xlims=[None,None], xticks=[None,list(CONVERGENCE['budgets'])],
    ylims=[None,None], yticks=[None,None], grid=False,    
    simulation=dict(marker='o', markersize=4, #markerfacecolor='none',
                    linestyle='None', capsize=2, elinewidth=1., markeredgewidth=1.),
    theory=dict(color='black',linewidth=1.8,linestyle='-',label='Theory'),
    budget_styles={}, budget_labels={},
    mse_line=dict(marker='o',markersize=4,linewidth=1.8,linestyle='-'),
    r_styles={}, r_labels={},
    show_legends=[True,True],
    legends=[dict(loc='upper left',fontsize=11,frameon=True,title=''),
             dict(loc='best',fontsize=11,frameon=True,title='')],
    annotation=None, annotation_xy=(.96,.96),  # None: stored lambda; False: hide
    annotation_style=dict(ha='right',va='top',fontsize=11,zorder=10,
                          bbox=dict(facecolor='white',edgecolor='none',alpha=.9,pad=2)),
)
CONVERGENCE_FORCE_REDRAW = False


In [ ]:
convergence_fig = plot_convergence(
    CONVERGENCE_PLOT, force=CONVERGENCE_FORCE_REDRAW)


## 7. Save the existing PDF

`PDF_FIGURES` maps the displayed convergence figure to its PDF filename.
Export never evaluates theory or simulates trials.


In [ ]:
# Save the displayed figure only; no numerical work is started.
PDF_FIGURES = {'convergence': 'appendix_finite_size.pdf'}
missing = [name for name in PDF_FIGURES if name not in FIGURES]
if missing:
    raise RuntimeError(f'Display these figures before export: {missing}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, filename in PDF_FIGURES.items():
    FIGURES[name].savefig(OUTPUT_DIR / filename, format='pdf', bbox_inches='tight')
    print(OUTPUT_DIR / filename)
